In [21]:
!nvidia-smi

Sat May  9 14:37:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   58C    P0             28W /   70W |     117MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
!pip install pycuda

In [23]:
import numpy as np
import time

In [24]:
N = 1000000

a = np.random.randn(N).astype(np.float32)
b = np.random.randn(N).astype(np.float32)

print("Arrays created successfully.")

Arrays created successfully.


In [25]:
start = time.time()

c_cpu = a + b

end = time.time()

cpu_time = end - start

print("CPU Time:", cpu_time)

CPU Time: 0.0014870166778564453


In [26]:
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule

In [27]:
mod = SourceModule("""
__global__ void add_vectors(float *dest, float *a, float *b)
{
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    dest[idx] = a[idx] + b[idx];
}
""")

In [28]:
a_gpu = cuda.mem_alloc(a.nbytes)
b_gpu = cuda.mem_alloc(b.nbytes)
c_gpu = cuda.mem_alloc(a.nbytes)

In [29]:
cuda.memcpy_htod(a_gpu, a)
cuda.memcpy_htod(b_gpu, b)

In [30]:
func = mod.get_function("add_vectors")

block_size = 256
grid_size = (N + block_size - 1) // block_size

start = time.time()

func(
    c_gpu,
    a_gpu,
    b_gpu,
    block=(block_size, 1, 1),
    grid=(grid_size, 1)
)

end = time.time()

gpu_time = end - start

print("GPU Time:", gpu_time)

GPU Time: 0.00016188621520996094


In [31]:
c_result = np.empty_like(a)

cuda.memcpy_dtoh(c_result, c_gpu)

In [32]:
print("Results match:", np.allclose(c_cpu, c_result))

Results match: True
